# 🌐 Week 11-2: Amazon CloudFront 배포 구성 및 글로벌 CDN 실습 (최적화)

### 🎯 실습 목표
- S3를 Origin으로 CloudFront 배포 구성
- OAC(Origin Access Control)로 S3 비공개 보안 강화
- 캐시 동작(Miss/Hit) 검증 및 캐시 무효화(Invalidation) 실습

### 🔑 사전 준비 — Colab Secrets 등록
좌측 🔑 아이콘 → **Secrets** 에 아래 2개 추가:

| 이름 | 값 |
|------|----|
| `AWS_ACCESS_KEY` | AKIA... |
| `AWS_SECRET_KEY` | 시크릿키 |

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 0: 패키지 설치 & 인증
# ══════════════════════════════════════════════════════════
!pip install boto3 -q

import boto3, json, time, os
from botocore.exceptions import ClientError
from google.colab import userdata

# ── Colab Secrets 로드 ────────────────────────────────────
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_KEY')
REGION                = 'ap-northeast-2'

def make_client(service):
    return boto3.client(
        service,
        aws_access_key_id     = AWS_ACCESS_KEY_ID,
        aws_secret_access_key = AWS_SECRET_ACCESS_KEY,
        region_name           = REGION,
    )

s3  = make_client('s3')
cf  = boto3.client(          # CloudFront는 글로벌 서비스 → region 불필요
    'cloudfront',
    aws_access_key_id     = AWS_ACCESS_KEY_ID,
    aws_secret_access_key = AWS_SECRET_ACCESS_KEY,
    region_name           = 'us-east-1',
)
sts = make_client('sts')

def ok(msg):   print(f'✅ {msg}')
def warn(msg): print(f'⚠️  {msg}')
def err(msg):  print(f'❌ {msg}')

# ── 인증 확인 ─────────────────────────────────────────────
identity   = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']
ok(f'인증 완료 — Account: {ACCOUNT_ID}, Region: {REGION}')

# ── 전역 변수 ─────────────────────────────────────────────
BUCKET_NAME = f'cloudarchitect-lab-s3website-{ACCOUNT_ID}'
OAC_NAME    = f'OAC-{BUCKET_NAME}'
DIST_COMMENT = 'CloudArchitect-Lab-Distribution'

print(f'   버킷명: {BUCKET_NAME}')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 1: S3 버킷 생성 & 파일 업로드
# ══════════════════════════════════════════════════════════

# ── 1-1. 버킷 생성 (이미 있으면 재사용) ──────────────────
try:
    s3.create_bucket(
        Bucket                    = BUCKET_NAME,
        CreateBucketConfiguration = {'LocationConstraint': REGION},
    )
    ok(f'S3 버킷 생성: {BUCKET_NAME}')
except ClientError as e:
    if e.response['Error']['Code'] == 'BucketAlreadyOwnedByYou':
        ok(f'S3 버킷 재사용: {BUCKET_NAME}')
    else:
        raise

# ── 1-2. 버킷 태그 설정 ──────────────────────────────────
s3.put_bucket_tagging(
    Bucket    = BUCKET_NAME,
    Tagging   = {'TagSet': [
        {'Key': 'Project',     'Value': 'CloudArchitect'},
        {'Key': 'Week',        'Value': 'Week11'},
        {'Key': 'Component',   'Value': 'Storage'},
        {'Key': 'Environment', 'Value': 'Lab'},
    ]},
)

# ── 1-3. 초기 퍼블릭 액세스 허용 (OAC 적용 전 임시) ──────
s3.put_public_access_block(
    Bucket                          = BUCKET_NAME,
    PublicAccessBlockConfiguration  = {
        'BlockPublicAcls':       False,
        'IgnorePublicAcls':      False,
        'BlockPublicPolicy':     False,
        'RestrictPublicBuckets': False,
    },
)
s3.put_bucket_policy(
    Bucket = BUCKET_NAME,
    Policy = json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Sid':       'PublicReadGetObject',
            'Effect':    'Allow',
            'Principal': '*',
            'Action':    's3:GetObject',
            'Resource':  f'arn:aws:s3:::{BUCKET_NAME}/*',
        }],
    }),
)

# ── 1-4. HTML 파일 생성 & 업로드 ────────────────────────
INDEX_HTML = '''\
<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <title>CloudArchitect Week11 - CloudFront</title>
  <style>
    body{font-family:Arial;text-align:center;background:#74b9ff;color:#fff;padding:60px}
    .box{max-width:700px;margin:0 auto;background:rgba(0,0,0,.2);border-radius:16px;padding:40px}
    h1{font-size:2.4rem;margin-bottom:.5rem}
    p{font-size:1.1rem;opacity:.9}
  </style>
</head>
<body>
  <div class="box">
    <h1>🌐 CloudArchitect Week11</h1>
    <h2>CloudFront CDN 배포 성공!</h2>
    <p>✅ S3 정적 웹사이트 + CloudFront CDN</p>
    <p>전 세계 어디서나 빠른 콘텐츠 전송!</p>
  </div>
</body>
</html>'''

ERROR_HTML = '''\
<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <title>Error - CloudArchitect Week11</title>
  <style>
    body{font-family:Arial;text-align:center;background:#e17055;color:#fff;padding:60px}
  </style>
</head>
<body>
  <h1>❌ 페이지를 찾을 수 없습니다</h1>
  <p>CloudArchitect Week11 - CloudFront 실습</p>
</body>
</html>'''

for key, body in [('index.html', INDEX_HTML), ('error.html', ERROR_HTML)]:
    s3.put_object(
        Bucket      = BUCKET_NAME,
        Key         = key,
        Body        = body.encode(),
        ContentType = 'text/html; charset=utf-8',
    )
    ok(f'업로드: s3://{BUCKET_NAME}/{key}')

ok('STEP 1 완료 — S3 버킷 & 파일 준비')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 2: OAC 생성 (또는 재사용) & CloudFront 배포 생성
# ══════════════════════════════════════════════════════════

# ── 2-1. OAC 검색 또는 생성 ──────────────────────────────
OAC_ID = None
paginator = cf.get_paginator('list_origin_access_controls')
for page in paginator.paginate():
    for item in page.get('OriginAccessControlList', {}).get('Items', []):
        if item['Name'] == OAC_NAME:
            OAC_ID = item['Id']
            ok(f'기존 OAC 재사용: {OAC_ID}')
            break
    if OAC_ID:
        break

if not OAC_ID:
    resp   = cf.create_origin_access_control(
        OriginAccessControlConfig={
            'Name':                          OAC_NAME,
            'Description':                   'OAC for Lab S3 Bucket',
            'SigningProtocol':               'sigv4',
            'SigningBehavior':               'always',
            'OriginAccessControlOriginType': 's3',
        }
    )
    OAC_ID = resp['OriginAccessControl']['Id']
    ok(f'OAC 생성 완료: {OAC_ID}')

# ── 2-2. 기존 배포 검색 (중복 방지) ─────────────────────
DIST_ID = DIST_DOMAIN = DIST_ARN = None
for page in cf.get_paginator('list_distributions').paginate():
    for d in page.get('DistributionList', {}).get('Items', []):
        if d.get('Comment') == DIST_COMMENT and d.get('Enabled'):
            DIST_ID     = d['Id']
            DIST_DOMAIN = d['DomainName']
            DIST_ARN    = d['ARN']
            ok(f'기존 배포 재사용: {DIST_ID}')
            break
    if DIST_ID:
        break

# ── 2-3. 배포 생성 ────────────────────────────────────────
if not DIST_ID:
    print('🔄 CloudFront 배포 생성 중...')
    resp = cf.create_distribution(
        DistributionConfig={
            'CallerReference':   str(int(time.time())),
            'Comment':           DIST_COMMENT,
            'Enabled':           True,
            'DefaultRootObject': 'index.html',
            'Origins': {
                'Quantity': 1,
                'Items': [{
                    'Id':         f'S3-{BUCKET_NAME}',
                    'DomainName': f'{BUCKET_NAME}.s3.{REGION}.amazonaws.com',
                    'S3OriginConfig':       {'OriginAccessIdentity': ''},
                    'OriginAccessControlId': OAC_ID,
                }],
            },
            'DefaultCacheBehavior': {
                'TargetOriginId':        f'S3-{BUCKET_NAME}',
                'ViewerProtocolPolicy':  'redirect-to-https',
                'TrustedSigners':        {'Quantity': 0, 'Enabled': False},
                'TrustedKeyGroups':      {'Quantity': 0, 'Enabled': False},
                'ForwardedValues': {
                    'QueryString':        False,
                    'Cookies':            {'Forward': 'none'},
                    'Headers':            {'Quantity': 0},
                    'QueryStringCacheKeys': {'Quantity': 0},
                },
                'MinTTL':     0,
                'DefaultTTL': 86400,
                'MaxTTL':     31536000,
            },
            'CustomErrorResponses': {
                'Quantity': 1,
                'Items': [{
                    'ErrorCode':         403,
                    'ResponsePagePath':  '/error.html',
                    'ResponseCode':      '403',
                    'ErrorCachingMinTTL': 10,
                }],
            },
        }
    )
    dist    = resp['Distribution']
    DIST_ID     = dist['Id']
    DIST_DOMAIN = dist['DomainName']
    DIST_ARN    = dist['ARN']
    ok(f'CloudFront 배포 생성 완료: {DIST_ID}')

print(f'\n   배포 ID    : {DIST_ID}')
print(f'   도메인 주소 : https://{DIST_DOMAIN}')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 3: S3 버킷 정책 → OAC 전용 접근으로 잠금
# ══════════════════════════════════════════════════════════

# 퍼블릭 액세스 완전 차단
s3.put_public_access_block(
    Bucket                         = BUCKET_NAME,
    PublicAccessBlockConfiguration = {
        'BlockPublicAcls':       True,
        'IgnorePublicAcls':      True,
        'BlockPublicPolicy':     True,
        'RestrictPublicBuckets': True,
    },
)

# CloudFront OAC 전용 버킷 정책 적용
s3.put_bucket_policy(
    Bucket = BUCKET_NAME,
    Policy = json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Sid':       'AllowCloudFrontServicePrincipalReadOnly',
            'Effect':    'Allow',
            'Principal': {'Service': 'cloudfront.amazonaws.com'},
            'Action':    's3:GetObject',
            'Resource':  f'arn:aws:s3:::{BUCKET_NAME}/*',
            'Condition': {
                'StringEquals': {'AWS:SourceArn': DIST_ARN}
            },
        }],
    }),
)

ok('STEP 3 완료 — S3 퍼블릭 차단 & OAC 단독 접근 정책 적용')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 4: 배포 완료 대기 & 캐시 동작 검증 (Miss → Hit)
# ══════════════════════════════════════════════════════════
import urllib.request

print('⏳ CloudFront 배포 완료(Deployed) 대기 중 (약 5~10분)...')
start = time.time()
while True:
    status = cf.get_distribution(Id=DIST_ID)['Distribution']['Status']
    elapsed = int(time.time() - start)
    print(f'   [{elapsed:3d}s] 상태: {status}')
    if status == 'Deployed':
        ok('배포 완료!')
        break
    time.sleep(20)

TARGET_URL = f'https://{DIST_DOMAIN}/index.html'
print(f'\n🌐 테스트 URL: {TARGET_URL}')

def check_cache(label):
    req  = urllib.request.Request(TARGET_URL)
    resp = urllib.request.urlopen(req)
    x_cache  = resp.headers.get('X-Cache', 'N/A')
    via      = resp.headers.get('Via',     'N/A')
    cf_pop   = resp.headers.get('X-Amz-Cf-Pop', 'N/A')
    print(f'  {label}')
    print(f'    X-Cache        : {x_cache}')
    print(f'    Via            : {via}')
    print(f'    X-Amz-Cf-Pop   : {cf_pop}')

print('\n--- [1차 요청: 캐시 Miss 예상] ---')
check_cache('1차')
time.sleep(2)
print('\n--- [2차 요청: 캐시 Hit 예상] ---')
check_cache('2차')

ok('STEP 4 완료 — 캐시 동작 검증')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 5: 콘텐츠 수정 & 캐시 무효화(Invalidation)
# ══════════════════════════════════════════════════════════

UPDATED_HTML = '''\
<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <title>수정본 - CloudFront 실습</title>
  <style>
    body{font-family:Arial;text-align:center;background:#00b894;color:#fff;padding:60px}
    .box{max-width:700px;margin:0 auto;background:rgba(0,0,0,.2);border-radius:16px;padding:40px}
  </style>
</head>
<body>
  <div class="box">
    <h1>🚀 Hello CloudFront!</h1>
    <p>캐시 무효화(Invalidation) 실습 성공!</p>
    <p>업데이트 시각: ''' + time.strftime('%Y-%m-%d %H:%M:%S UTC') + '''</p>
  </div>
</body>
</html>'''

# 1. S3에 수정본 업로드
s3.put_object(
    Bucket      = BUCKET_NAME,
    Key         = 'index.html',
    Body        = UPDATED_HTML.encode(),
    ContentType = 'text/html; charset=utf-8',
)
ok('수정된 index.html → S3 업로드 완료')

# 2. 무효화 전 캐시 확인 (구버전 캐시 노출)
print('\n--- [무효화 이전: 캐시 구버전 응답 확인] ---')
check_cache('무효화 전')

# 3. 캐시 무효화 요청
print('\n🔄 캐시 무효화(Invalidation) 요청 중...')
inval = cf.create_invalidation(
    DistributionId    = DIST_ID,
    InvalidationBatch = {
        'Paths':           {'Quantity': 1, 'Items': ['/index.html']},
        'CallerReference': str(int(time.time())),
    },
)
INVAL_ID = inval['Invalidation']['Id']
ok(f'무효화 요청 생성: {INVAL_ID}')

# 4. 무효화 완료 대기
print('⏳ 무효화 완료 대기 중...')
while True:
    status = cf.get_invalidation(
        DistributionId = DIST_ID, Id = INVAL_ID
    )['Invalidation']['Status']
    print(f'   무효화 상태: {status}')
    if status == 'Completed':
        ok('무효화 완료!')
        break
    time.sleep(10)

# 5. 최종 확인 (신버전 응답)
print('\n--- [무효화 완료 후: 신버전 응답 확인] ---')
check_cache('무효화 후')

print(f'\n🌐 최종 접속 URL: https://{DIST_DOMAIN}')
ok('STEP 5 완료 — 콘텐츠 수정 & 캐시 무효화 성공')

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 6: 리소스 정리 (Cleanup) — 과금 방지
# ══════════════════════════════════════════════════════════
# ⚠️ 실습 완료 후에만 실행하세요!

print('🗑️  리소스 정리를 시작합니다.')

# 1. CloudFront 배포 비활성화
print('\n1. CloudFront 배포 비활성화 중...')
dist_resp  = cf.get_distribution_config(Id=DIST_ID)
etag       = dist_resp['ETag']
dist_cfg   = dist_resp['DistributionConfig']
dist_cfg['Enabled'] = False

cf.update_distribution(
    Id                 = DIST_ID,
    DistributionConfig = dist_cfg,
    IfMatch            = etag,
)
ok('비활성화 업데이트 완료. 글로벌 반영 대기 중...')

while True:
    status = cf.get_distribution(Id=DIST_ID)['Distribution']['Status']
    print(f'   상태: {status}')
    if status == 'Deployed':
        ok('비활성화 완료')
        break
    time.sleep(20)

# 2. CloudFront 배포 삭제
final_etag = cf.get_distribution(Id=DIST_ID)['ETag']
cf.delete_distribution(Id=DIST_ID, IfMatch=final_etag)
ok('CloudFront 배포 삭제 완료')

# 3. OAC 삭제
oac_etag = cf.get_origin_access_control(Id=OAC_ID)['ETag']
cf.delete_origin_access_control(Id=OAC_ID, IfMatch=oac_etag)
ok('OAC 삭제 완료')

# 4. S3 버킷 객체 삭제 후 버킷 삭제
print('\n2. S3 버킷 정리 중...')
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET_NAME):
    objects = [{'Key': o['Key']} for o in page.get('Contents', [])]
    if objects:
        s3.delete_objects(Bucket=BUCKET_NAME, Delete={'Objects': objects})

s3.delete_bucket(Bucket=BUCKET_NAME)
ok('S3 버킷 삭제 완료')

print('\n🎉 모든 실습 자원이 안전하게 정리되었습니다!')